# 使用 PROC COMPUTAB 编制季度预估损益表

## 执行摘要

本笔记本使用 SAS/ETS 报表编制表过程 PROC COMPUTAB，直接根据每月分类账数据构建一家区域性银行的季度预估损益表。它将每个月的利息收入、利息支出、手续费收入和运营成本路由到正确的日历季度列中，然后使用行编程块计算净利息收入、税前收入和净收入，并使用列块将四个季度汇总为全年合计。

## 数据来源

单一的合成数据集 `bank_ledger` 模拟一家中型社区银行一个财政年度的每月财务报表项目。十二个月度观测值通过 `call streaminit`/`rand` 内联生成，因此本笔记本完全自包含。

| 变量 | 类型 | 描述 |
|----------|------|-------------|
| `stmtdate` | 数值 (DATE9.) | 月末报表日期（1月–12月） |
| `loanint`  | 数值 | 贷款组合赚取的利息和手续费（千美元） |
| `secint`   | 数值 | 投资证券账簿赚取的利息（千美元） |
| `depint`   | 数值 | 存款支付的利息（千美元） |
| `borrint`  | 数值 | 借款 / FHLB 预支款支付的利息（千美元） |
| `feeinc`   | 数值 | 非利息（手续费及服务收费）收入（千美元） |
| `salaries` | 数值 | 薪金和员工福利支出（千美元） |
| `occupancy`| 数值 | 房产占用和设备支出（千美元） |
| `othropex` | 数值 | 其他非利息运营支出（千美元） |
| `provision`| 数值 | 信贷损失准备金（千美元） |
| `taxrate`  | 数值 | 应用于税前收入的实际税率 |

# 使用 PROC COMPUTAB 编制季度预估损益表

银行财务团队经常将每月总账转换为显示净利息收入和最终净收入的**季度损益表**。`PROC COMPUTAB`（SAS/ETS）专为此而设计：它布局一个可编程的表，其中**列**为报告期间，**行**为报表项目，并允许你在行块和列块中使用普通的 DATA step 逻辑来计算小计。

在本笔记本中，我们：

1. 为一家社区银行生成一个财政年度的合成月度分类账数据。
2. 将每个月路由到其所属的日历季度，并构建季度损益表。
3. 在**行块**中计算净利息收入、税前收入和净收入，并在**列块**中将各季度汇总为全年合计。
4. 复用 `out=` 表，使计算出的报表可以供下游报告使用。

## 步骤 1 — 生成合成月度分类账数据

我们模拟十二个月末观测值。贷款和证券利息收入在全年中平缓上升，存款和借款成本随利率环境变化，手续费收入、运营支出和信贷损失准备金带有真实的月度间噪声。`call streaminit` 固定种子以使数据可复现。

In [1]:
数据 bank_ledger;
   调用 streaminit(20240115);
   格式 stmtdate date9.;
   循环 month = 1 到 12;
      /* Month-end statement date for fiscal year 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Mild upward drift across the year + monthly noise */
      drift = 1 + 0.012 * (month - 1);

      /* Interest income (USD thousands) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Interest expense (USD thousands) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Non-interest income and expense (USD thousands) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Provision for credit losses, occasionally elevated */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Effective tax rate */
      taxrate = 0.21;

      输出;
   结束;
   保留 stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
运行;

过程 打印 数据=bank_ledger noobs 标签;
   标题 'Synthetic Monthly Ledger — Community Bank FY2024';
   格式 loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
运行;

                                    Synthetic Monthly Ledger — Community Bank FY2024                                    

 STMTDATE   LOANINT  SECINT  DEPINT  BORRINT  FEEINC  SALARIES  OCCUPANCY  OTHROPEX  PROVISION  TAXRATE
28JAN2024  1,772.95  423.79  531.47   128.99  339.33    699.38     171.95    202.43     130.93     0.21
28FEB2024  1,824.38  421.13  564.85   125.53  294.09    767.29     162.79    307.61     123.25     0.21
28MAR2024  1,931.98  442.06  536.64   131.71  295.72    724.03     153.26    254.21     115.76     0.21
28APR2024  1,934.99  439.29  535.94   140.01  294.56    729.47     174.19    237.43     198.05     0.21
28MAY2024  1,949.31  447.35  591.44   124.42  299.78    739.13     165.08    223.32     105.57     0.21
28JUN2024  1,934.36  448.28  551.70   147.64  335.81    718.90     171.91    236.94     130.13     0.21
28JUL2024  1,936.57  439.22  565.70   133.82  293.21    743.87     174.16    247.19     199.20     0.21
28AUG2024  1,979.73  457.42  558.54   144.45  

## 步骤 2 — 构建季度损益表

报表的核心是下面的 `PROC COMPUTAB` 步。

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** 定义四个季度列加上一个全年列。
- 未加标签的**输入块**使用 `qtr(stmtdate)` 设置自动变量 **`_col_`**，从而将每个月度观测值路由到正确的季度列。由于输入默认被转置，每个分类账变量都落在与其同名的行上。
- **行块** `inc_stmt:` 对每一列运行一次，计算派生的项目——净利息收入、非利息支出合计、税前收入和净收入——完全如同会计师所做的那样。
- **列块** `total:` 对每一行运行一次，将四个季度加总到 `fy`（全年）列中。

`rows ... / ...` 语句添加标题、缩进和分隔线（`ol` 上划线、`ul` 下划线、`dul` 双下划线），使输出读起来像一份真实的财务报表。

In [2]:
标题 'Pro Forma Income Statement — Community Bancorp, Inc.';
title2 'Fiscal Year 2024  (Amounts in USD Thousands)';

过程 computab 数据=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Four quarters plus a rolled-up full-year column */
   columns qtr1 qtr2 qtr3 qtr4 fy / 格式=comma11.1;
   columns qtr1 / 'Q1';
   columns qtr2 / 'Q2';
   columns qtr3 / 'Q3';
   columns qtr4 / 'Q4';
   columns fy   / 'Full' 'Year' +3;

   /* Income statement rows, top to bottom */
   rows loanint  / 'Interest & Fees on Loans';
   rows secint   / 'Interest on Securities'        ul;
   rows intinc   / 'Total Interest Income';
   rows depint   / 'Interest on Deposits';
   rows borrint  / 'Interest on Borrowings'        ul;
   rows intexp   / 'Total Interest Expense';
   rows nii      / 'Net Interest Income'           ol skip;
   rows provision/ 'Provision for Credit Losses'   ul;
   rows niiap    / 'Net Int. Income after Prov.'   skip;
   rows feeinc   / 'Non-Interest Income'           ul;
   rows salaries / 'Salaries & Benefits';
   rows occupancy/ 'Occupancy & Equipment';
   rows othropex / 'Other Operating Expense'       ul;
   rows nonintexp/ 'Total Non-Interest Expense'    skip;
   rows pretax   / 'Pre-Tax Income'                ol;
   rows taxexp   / 'Income Tax Expense'            ul;
   rows netinc   / 'Net Income'                    dul;

   /* Input block: route each month to its calendar quarter */
   _col_ = qtr(stmtdate);

   /* Row block: compute statement subtotals across every column */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Column block: roll the four quarters into the full-year column */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
运行;

                                  Pro Forma Income Statement — Community Bancorp, Inc.                                  
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


                             The COMPUTAB Procedure                             

                             Q1           Q2           Q3           Q4         Full  
                                                                               Year  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Interest & Fees on Loans
  loanint               5529.31      5818.66      5963.46      6276.96     23588.39  
  Interest on Securities
  secint                1286.98      1334.92      1342.03      1452.88      5416.81  
                    -----------  -----------  -----------  -----------  -----------  
  Total Interest Inc

## 步骤 3 — 复用 COMPUTAB 输出数据集

上面的 `PROC COMPUTAB` 步通过 `out=` 将其计算出的表写入 `qtr_income`。该数据集的每一行是一个报表项目，每个列变量（`qtr1`–`qtr4`、`fy`）保存计算出的值，因此可供下游报告使用。下面我们打印汇总后的全年列，以确认数字对得上。

In [3]:
标题 'COMPUTAB Output Dataset — Full-Year Column';

过程 打印 数据=qtr_income 标签 noobs;
   变量 _row_ fy;
   格式 fy comma12.1;
   标签 _row_ = 'Line Item' fy = 'Full Year';
运行;

标题;

                                       COMPUTAB Output Dataset — Full-Year Column                                       
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


Line Item  Full Year
---------  ---------
loanint     23,588.4
secint       5,416.8
intinc      29,005.2
depint       6,831.2
borrint      1,650.7
intexp       8,482.0
nii         20,523.2
provision    1,568.9
niiap       18,954.3
feeinc       3,703.2
salaries     8,763.1
occupancy    1,985.6
othropex     2,944.8
nonintexp   13,693.5
pretax       8,964.1
taxexp       1,882.5
netinc       7,081.6

NOTE: Option TITLE changed to COMPUTAB Output Dataset — Full-Year Column.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## 结果解读

这份预估报表自上而下阅读，就像银行的监管报告（call report）：总利息收入减去总利息支出得出**净利息收入**——此处全年约为 \$20.5 百万——这是银行的主要盈利驱动力。减去**信贷损失准备金**，加上**手续费收入**，并扣除**非利息支出**，得出约 \$9.0 百万的税前收入，再应用 21% 的实际税率产生接近 \$7.1 百万的**净收入**。`total:` 列块将四个季度加总到全年列中，因此年度合计从构造上就与各季度之和对账一致——这在 `out=` 数据集中得到了确认，其中全年 `netinc` 为 7,081.6，等于四个季度数字之和。

由于所有内容都在 `PROC COMPUTAB` 的可编程行块和列块内计算，替换成真实的月度分类账无需更改报表逻辑——只需更改输入数据集。然后 `out=` 数据集使计算出的报表可用于仪表板、趋势分析或董事会资料包自动化。